# 01 · Descubrimiento y Transporte

**Objetivo del notebook**: entender cómo dos procesos sin servidor central se *encuentran* y mantienen un canal fiable.

Cubre las capas 1 y 2 de [p2p-chat](../ARCHITECTURE.md). Lee los fuentes:
[src/discovery.ts](../../src/discovery.ts) y [src/transport.ts](../../src/transport.ts) en paralelo.

## 1. El problema

En una red P2P **no hay servidor central** que mantenga el directorio de quién está conectado. Cada peer es a la vez:

- cliente (inicia conexiones a peers descubiertos)
- servidor (acepta conexiones entrantes)
- directorio (responde preguntas sobre lo que sabe)

Pero antes de hablar con alguien hace falta **saber que existe**. Ese es el problema del *peer discovery*.

## 2. Descubrimiento por UDP broadcast

Mecanismo más simple en LAN: cada peer **grita su existencia** periódicamente a la dirección de broadcast (`255.255.255.255`). Todos los demás peers escuchan ese puerto.

### Anatomía de un anuncio

Cada 3 segundos, cada peer envía un datagrama UDP con este JSON:

```json
{
  "peerId": "9bc9babb01663a13...",
  "tcpHost": "0.0.0.0",
  "tcpPort": 53282,
  "v": 1
}
```

**Por qué cada campo**:

- `peerId`: identidad efímera (hex de SHA-256 de 32 bytes aleatorios). No es criptográfica — cualquiera podría suplantar uno — pero es suficiente para distinguir peers en una sesión.
- `tcpHost = "0.0.0.0"`: el receptor IGNORA este campo. La IP real del peer se extrae de `rinfo.address` que el SO añade al recibir el datagrama. Esto evita que un peer mienta sobre su propia IP.

  > **¿Por qué existe `tcpHost` si siempre se ignora?**
  >
  > Tres razones de diseño de protocolo:
  >
  > 1. **Contrato de schema**: el JSON siempre tiene `{peerId, tcpHost, tcpPort, v}`. Un parser que espera esos campos no explota ante un campo ausente. Presencia explícita = contrato cumplido.
  >
  > 2. **Caso edge — NAT doble / VPN**: si el peer está detrás de NAT simétrico o VPN, `rinfo.address` devuelve la IP del router o del nodo VPN, no la IP interna útil para TCP. En ese escenario, el peer podría poner su IP interna real en `tcpHost` y el receptor *elegiría* usarla. Hoy no implementado, pero el campo deja la puerta abierta.
  >
  > 3. **Caso edge — host multi-interfaz**: una máquina con WiFi + Ethernet puede enviar el datagrama UDP por una interfaz pero escuchar TCP en la otra. `tcpHost` permite indicar explícitamente la interfaz TCP correcta.
  >
  > **En LAN simple (caso actual):** `"0.0.0.0"` es señal explícita de *"no uses este campo, confía en `rinfo.address`"*. El receptor siempre gana.

- `tcpPort`: por dónde recibe TCP (el descubrimiento es UDP; el chat real va por TCP).
- `v`: versión del protocolo de descubrimiento; permite cambiar el formato sin romper.

### Limitaciones del broadcast

| Limitación | Consecuencia |
|------------|--------------|
| No sale del router | Solo funciona en la misma LAN |
| Tráfico se multiplica con N peers | Para N=1000+ ya es ruidoso |
| Sin autenticación | Cualquiera en LAN puede suplantar peerId |

Para WAN harían falta DHT (Kademlia), trackers o mDNS sobre VPN. Ver [docs/NAT.md](../../../docs/NAT.md).

## 3. Tabla de peers vivos + GC

Mantener "quién está vivo" no es trivial sin un servidor. Solución pragmática:

1. Cada vez que llega un anuncio, **actualizamos `lastSeen[peerId] = now`**.
2. Cada 2 s recorremos la tabla y borramos los peers con `lastSeen < now - 10 s`.

Es decir: si pasan **3 ciclos de anuncio sin oír** a un peer, lo damos por muerto. El timeout de 10s sobre intervalo de 3s da margen para pérdidas de paquetes UDP (UDP no garantiza entrega).

Vamos a simular el comportamiento:

In [ ]:
import time
import random

ANNOUNCE_INTERVAL = 3.0  # s
PEER_TIMEOUT = 10.0      # s
PACKET_LOSS = 0.2        # 20% paquetes perdidos

peers = {}  # peerId -> last_seen

def announce_received(peer_id, t):
    if random.random() < PACKET_LOSS:
        return  # paquete perdido
    peers[peer_id] = t

def gc(t):
    dead = [p for p, ts in peers.items() if t - ts > PEER_TIMEOUT]
    for p in dead:
        del peers[p]
    return dead

# Simulación: peer 'B' anuncia cada 3s durante 30s, luego se calla
for t in range(0, 50):
    if t < 30 and t % 3 == 0:
        announce_received('B', t)
    dead = gc(t)
    if dead:
        print(f't={t:02d}s  GC dropped: {dead}')
    if t % 5 == 0:
        print(f't={t:02d}s  alive: {list(peers)}')


### Variante con librería: `kademlia` (DHT — sale de LAN)

```
pip install kademlia
```

El broadcast UDP solo alcanza el router local. **Kademlia** es una DHT (Distributed Hash Table): cada peer conoce al menos un nodo semilla (*bootstrap*), se integra al anillo y puede anunciar/descubrir peers **a través de internet**. El mismo par `(peerId → tcpPort)` que antes vivía en el broadcast ahora vive distribuido en el DHT.

```python
from kademlia.network import Server
import asyncio

async def peer(port, bootstrap=None):
    server = Server()
    await server.listen(port)

    if bootstrap:
        await server.bootstrap([bootstrap])        # une al anillo DHT

    peer_id = "9bc9babb01663a13"
    await server.set(peer_id, str(port + 1000))    # anuncio: peerId → tcpPort
    tcp_port = await server.get(peer_id)            # descubrimiento
    print(f"peer en TCP:{tcp_port}")
    server.stop()

# Ejecutar en dos máquinas distintas (o dos puertos locales para probar):
# Máquina A (bootstrap):  asyncio.run(peer(5678))
# Máquina B (joiner):     asyncio.run(peer(5679, bootstrap=('IP_de_A', 5678)))
```

**Diferencias clave vs broadcast UDP**:

| | Broadcast UDP (implementación propia) | Kademlia (`kademlia`) |
|---|---|---|
| Alcance | Solo LAN | Internet |
| Implementación | ~60 líneas TS | 3 líneas Python |
| GC de peers muertos | Manual — timeout 10 s | TTL configurable en el DHT |
| Escalabilidad | O(N) tráfico broadcast | O(log N) saltos de lookup |
| Nodo de entrada | Ninguno (todos gritan) | Bootstrap node (cualquier peer conocido) |

El GC de la simulación anterior desaparece: el DHT expira claves automáticamente por TTL. La desventaja: necesitas al menos un bootstrap node accesible. En producción esto suele ser un nodo fijo (tipo servidor de arranque de BitTorrent) o una lista de nodos conocidos hardcodeada.

## 4. Transporte: ¿por qué TCP?

Una vez sabemos que B existe en `192.168.1.42:53282`, necesitamos un canal **fiable** y **ordenado** para mandar mensajes. Opciones:

| Protocolo | Pros | Contras |
|-----------|------|---------|
| **UDP** | Bajo overhead, mejor para tiempo real | Sin orden, sin retransmisiones, sin control de congestión |
| **TCP** | Fiable, ordenado, control de congestión nativo | Más latencia ante pérdida (head-of-line blocking) |
| **QUIC** | Lo bueno de TCP sin head-of-line | Implementación compleja |

Para **mensajería** (texto, ack, comandos): TCP es perfecto. Una retransmisión silenciosa de 50 ms en un mensaje de chat es invisible al usuario.

Para **audio en tiempo real** TCP es pésimo (eso lo veremos en el notebook 04: ahí WebRTC monta SRTP sobre UDP).

## 5. Pool de conexiones por peerId

Cada peer mantiene:

- Un `net.Server` TCP escuchando.
- Un `Map<peerId, Conn>` indexado por peerId, no por socket.

**Por qué indexar por peerId y no por socket**: el código de la aplicación quiere mandar mensajes a `peerId X`, no preguntarse "¿qué socket es de X?". Encapsulamos eso en el pool.

### El problema de los sockets duplicados

Cuando A y B se descubren a la vez, ambos intentan conectarse al otro:

```
A.discovery: peer-up(B) → A.connect(B)
B.discovery: peer-up(A) → B.connect(A)
```

Resultado: dos sockets entre A y B. Solución elegante: **tie-break lexicográfico**.

```
discovery.on('peer-up', (p) => {
  if (peerId < p.peerId) transport.connect(p.peerId, ...);
});
```

Solo el peerId menor inicia la conexión saliente. El mayor acepta entrantes. Determinista, sin coordinación.

## 6. Handshake HELLO

Un socket TCP recién aceptado **no sabe qué peer hay al otro lado** — solo conoce su IP. Resuelve esto el handshake:

1. Cada extremo manda inmediatamente un `HELLO { peerId, version }`.
2. Hasta no recibir un HELLO, el resto de mensajes se descartan (defensa contra mensajes pre-handshake).
3. Si el peerId del HELLO ya está en el pool → conexión duplicada, cerrar la nueva.
4. Si todo OK → promover de `pending` a `conns` y emitir `peer-connected`.

Diagrama temporal:

```
A                                        B
─                                        ─
net.connect(B:53282)              accept()
                                  send HELLO(B)       ◄── B no esperó
send HELLO(A)         ────►       parse HELLO(A)
                                  pool[A] = socket
                                  emit peer-connected
                      ◄────       parse HELLO(B)
pool[B] = socket
emit peer-connected
```

Ambos HELLOs se cruzan sin problemas porque cada extremo solo espera UN HELLO antes de promover.

## 7. Siguiente paso

Tenemos un canal TCP fiable entre cada par de peers. Pero TCP es un **stream de bytes**, no de mensajes. Si A envía dos veces 100 bytes, B puede leer 200 de golpe, o 80+120, o como el kernel decida.

Hace falta una capa que defina dónde acaba un mensaje y empieza el siguiente. Eso es el **framing**, en el notebook 02.